# 03 Collaborative Filtering Modeling

This notebook demonstrates how we build the user-track interaction matrix, inspect sparsity, and generate example collaborative recommendations.

The production logic stays in `src/models`. The notebook only orchestrates and visualizes those reusable classes.

## Setup

We add `src/` to the notebook path so the same project modules work cleanly in VS Code on Mac.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Point the notebook at the repository source tree.
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from models.collaborative_recommender import CollaborativeRecommender
from models.interaction_matrix import InteractionMatrixBuilder


## Example Interaction Data

In a real system, these interactions could come from playlist adds, likes, repeated listens, skips, or session-level engagement events that have been converted into an implicit strength value.

In [ ]:
# Create a small implicit-feedback table with overlapping taste signals.
interactions = pd.DataFrame(
    [
        {"user_id": "u1", "track_id": "track_1", "interaction_strength": 4.0},
        {"user_id": "u1", "track_id": "track_2", "interaction_strength": 3.0},
        {"user_id": "u2", "track_id": "track_2", "interaction_strength": 4.0},
        {"user_id": "u2", "track_id": "track_3", "interaction_strength": 2.0},
        {"user_id": "u3", "track_id": "track_2", "interaction_strength": 2.0},
        {"user_id": "u3", "track_id": "track_3", "interaction_strength": 4.0},
        {"user_id": "u3", "track_id": "track_4", "interaction_strength": 1.0},
        {"user_id": "u4", "track_id": "track_1", "interaction_strength": 1.0},
        {"user_id": "u4", "track_id": "track_4", "interaction_strength": 4.0},
    ]
)

interactions


## Build The Sparse Interaction Matrix

The matrix builder aggregates repeated user-track events and converts them into a CSR sparse matrix. This is the main collaborative-filtering input artifact.

In [ ]:
builder = InteractionMatrixBuilder()

# Build reusable sparse artifacts for training and serving.
artifacts = builder.build(interactions)
stats = builder.summarize(artifacts)

print(f"Users: {stats.num_users}")
print(f"Tracks: {stats.num_tracks}")
print(f"Observed interactions: {stats.num_interactions}")
print(f"Density: {stats.density:.4f}")
print(f"Sparsity: {stats.sparsity:.4f}")


In [ ]:
# Convert the sparse matrix to a small dense preview for teaching and debugging.
dense_preview = pd.DataFrame(
    artifacts.interaction_matrix.toarray(),
    index=artifacts.user_ids,
    columns=artifacts.track_ids,
)

dense_preview


## Train The Collaborative Model

The recommender learns item-item similarity from co-consumption patterns. Later, the hybrid recommender can blend these collaborative scores with content-based similarity scores to cover both taste overlap and musical attribute similarity.

In [ ]:
recommender = CollaborativeRecommender().fit(artifacts)

# Recommend tracks for a user based on their interaction history and similar users' behavior.
recommendations = recommender.recommend_for_user(user_id="u1", k=3)
recommendation_frame = pd.DataFrame(
    [
        {
            "track_id": item.item_id,
            "score": item.score,
            "source": item.source,
        }
        for item in recommendations
    ]
)

recommendation_frame


## Interview Note

Collaborative filtering helps because it can recommend tracks that do not look similar in raw audio features but are frequently enjoyed by similar listeners.

Its main limitation is cold start: new users and new tracks have little or no interaction history. In production, the fallback strategy should be a hybrid approach that leans on content-based recommendations and popularity-aware defaults until enough behavior is observed.